# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dyajaballh8/FlyRank_Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Unit of analysis

Each row represents daily performance data for one content item for one client.

### Time window

The daily performance data covers 2025-01-27 through 2026-06-30.

### Verification

The warehouse contains 78,835,655 rows covering 2025-01-27 through 2026-06-30. The grain check also revealed some repeated content-client-date combinations, so the data should not be assumed to have a perfectly unique grain.

In [20]:
from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "FlyRank/internship-warehouse"

print("Connection to warehouse is ready.")


Connection to warehouse is ready.


In [21]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files[:50]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [22]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_auth (
    TYPE HTTP,
    BEARER_TOKEN '{HF_TOKEN}'
)
""")

query = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
"""

result = con.execute(query).fetchdf()

result

grain_check = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_result = con.execute(grain_check).fetchdf()

grain_result

duplicate_check = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
WHERE report_date = '2026-06-13'
  AND client_hash_id = 'client_8ddc46da5414ffd8'
  AND content_hash_id = 'content_1d06a2c99a935e49'
"""

duplicate_rows = con.execute(duplicate_check).fetchdf()

duplicate_rows

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
1,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06


## 2. Fields: feature / label / context / excluded

### Feature

The daily performance metrics are candidate features when they are available before the prediction window:
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

### Label

The future performance outcome will be defined separately from the historical feature window. Current-day and future outcome measurements must not be used as features.

### Context

- report_date — used for time ordering and time-based splits.
- client_hash_id — used for grouping and client-level splits.
- content_hash_id — used for identifying and joining content items.
- client_has_gsc — indicates whether GSC exists for the client.
- client_has_ga4 — indicates whether GA4 exists for the client.
- gsc_data_available — indicates whether GSC data is available for the row.
- ga4_data_available — indicates whether GA4 data is available for the row.
- month — used for partitioning and time-based querying.

### Excluded

No additional fields are excluded at this stage. Any field that represents future information or is derived from the future outcome will be excluded when the prediction window and label are defined.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
columns = con.execute("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
""").fetchdf()

columns[["column_name", "column_type"]]


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [24]:
field_check = con.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
""").fetchdf()

field_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,clients,contents,min_date,max_date
0,78835655,70,427292,2025-01-27,2026-06-30


## 3. Verify it with queries (grain, counts, missing values, windows)
### Verification

The daily performance table contains 78,835,655 observed rows covering 2025-01-27 through 2026-06-30. It contains 70 pseudonymized clients and 427,292 content items.

The grain check identified repeated client-content-date combinations, so the table should not be assumed to have a perfectly unique one-row-per-client-content-day grain.

Missingness and data availability flags should be checked before using GSC or GA4 metrics as features. In particular, a missing or zero value should not automatically be interpreted as no activity because data availability differs across clients and dates.

The time window and field availability should be verified directly from the warehouse before feature construction or modeling.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
verification = con.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,

    SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END)
        AS rows_without_gsc,

    SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END)
        AS rows_without_ga4,

    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END)
        AS missing_gsc_impressions,

    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END)
        AS missing_ga4_sessions

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
""").fetchdf()

verification


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,clients,contents,min_date,max_date,rows_without_gsc,rows_without_ga4,missing_gsc_impressions,missing_ga4_sessions
0,78835655,70,427292,2025-01-27,2026-06-30,49767598.0,46383873.0,98006.0,29635327.0


## 4. Data limits

### Data limits

This dataset describes observed search and analytics performance, but it cannot explain why a metric changed or prove that one action caused the change.

Data availability is uneven across clients and dates. Some rows do not have GSC or GA4 data available, so missing values or zeros should not automatically be interpreted as no activity.

The dataset also contains repeated client-content-date combinations, so the grain is not perfectly unique without additional deduplication or aggregation rules.

The available history differs across clients, which means one global time window may not provide an equally long history for every client.

The final month of the warehouse should be treated carefully as an outcome or evaluation window. Feature windows must be aligned so that future information does not leak into the prediction process.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
limits_check = con.execute("""
SELECT
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT CASE
        WHEN gsc_data_available = TRUE THEN client_hash_id
    END) AS clients_with_gsc_rows,
    COUNT(DISTINCT CASE
        WHEN ga4_data_available = TRUE THEN client_hash_id
    END) AS clients_with_ga4_rows,

    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_0.parquet'
)
""").fetchdf()

limits_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_clients,clients_with_gsc_rows,clients_with_ga4_rows,earliest_date,latest_date
0,70,67,51,2025-01-27,2026-06-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.